Squad 2 | Streaming em Tempo Real

**Tabela** | ecommerce_produtos |

**Destino** | squad2.ecommerce_produtos |

**Schema** | sku, nome_produto, descricao, id_categoria, preco_lista, unidade_medida, nome_marca, is_ativo |

**Chave PK** | sku |

**Chave FK** | id_categoria → ecommerce_categorias |

**Nulos** | Nenhum |

**Depende de** | feat_squad2_99_helpers |

In [0]:
%run ../utils/feat_squad2_99_helpers

In [0]:
import logging
logging.getLogger("azure").setLevel(logging.WARNING)

TABELA        = "ecommerce_produtos"
MODO_GRAVACAO = "overwrite"

inicio = log_inicio(f"feat_squad2_04_ingestao_{TABELA}")

In [0]:
try:
    snapshot_id = get_snapshot_mais_recente()
    log.info(f"Snapshot selecionado: {snapshot_id}")

    df = ler_parquet(snapshot_id, TABELA)
    log.info(f"Leitura OK → {df.count()} linhas | {len(df.columns)} colunas")

except Exception as e:
    log.error(f"Erro ao ler {TABELA}: {str(e)}")
    raise

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, when

# Schema
log.info("Schema:")
df.printSchema()

# Amostra
display(df)

# Nulos
df_nulos = df.select([
    spark_sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df.columns
])

log.info("Nulos por coluna:")
display(df_nulos)

# Validações de negócio
total          = df.count()
total_ativos   = df.filter(col("is_ativo") == True).count()
total_inativos = df.filter(col("is_ativo") == False).count()
total_marcas   = df.select("nome_marca").distinct().count()
total_categorias = df.select("id_categoria").distinct().count()

log.info(f"Total registros  : {total}")
log.info(f"Produtos ativos  : {total_ativos}")
log.info(f"Produtos inativos: {total_inativos}")
log.info(f"Total marcas     : {total_marcas}")
log.info(f"Total categorias : {total_categorias}")


In [0]:
try:
    # Carrega categorias para validar FK
    df_categorias = ler_parquet(snapshot_id, "ecommerce_categorias")

    df_orfaos = df.join(
        df_categorias.select("id_categoria"),
        df["id_categoria"] == df_categorias["id_categoria"],
        "left_anti"
    )

    total_orfaos = df_orfaos.count()

    if total_orfaos == 0:
        log.info("Integridade referencial OK — todos os produtos têm categoria válida")
    else:
        log.warning(f" {total_orfaos} produto(s) com id_categoria inválido")
        display(df_orfaos)

except Exception as e:
    log.error(f"Erro na validação referencial: {str(e)}")

In [0]:
try:
    sucesso = gravar_sql(df, TABELA, mode=MODO_GRAVACAO)

    if sucesso:
        log.info(f"Gravação OK → {get_destino_sql(TABELA)}")
    else:
        raise Exception("Falha na gravação")

except Exception as e:
    log.error(f"Erro ao gravar {TABELA}: {str(e)}")
    raise

In [0]:
try:
    df_sql    = ler_sql(TABELA)
    total_sql = df_sql.count()

    log.info(f"Validação OK → {get_destino_sql(TABELA)}")
    log.info(f"Registros gravados: {total_sql}")

    if total_sql == total:
        log.info("✅ Origem e destino com mesmo número de registros!")
    else:
        log.warning(
            f"⚠️ Divergência: "
            f"origem={total} | destino={total_sql}"
        )

    display(df_sql)

except Exception as e:
    log.error(f"Erro na validação: {str(e)}")
    raise

In [0]:
log_fim(f"feat_squad2_04_ingestao_{TABELA}", inicio)